In [1]:
%cd ../..

c:\Users\ZunMoeThiri\OneDrive - Vena Group\Desktop\Folder - Forecasting\Modern-Time-Series-Forecasting-with-Python


In [2]:
import numpy as np
import os
import pandas as pd
from pathlib import Path
from tqdm.autonotebook import tqdm
import statsmodels.api as sm
import warnings
import random
from IPython.display import display, HTML
np.random.seed(42)
random.seed(42)
tqdm.pandas()

C:\Users\ZunMoeThiri\AppData\Local\Temp\ipykernel_15364\3033596466.py:5: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Reading and Selecting Households

In [3]:
# try:
#     lclid_acorn_map = pd.read_pickle("data/london_smart_meters/preprocessed/london_smart_meters_lclid_acorn_map.pkl")
# except FileNotFoundError:
#     display(HTML("""
#     <div class="alert alert-block alert-warning">
#     <b>Warning!</b> File not found. Please make sure you have run 02 - Preprocessing London Smart Meter Dataset.ipynb in Chapter02
#     </div>
#     """))

In [4]:
# affluent_households = lclid_acorn_map.loc[lclid_acorn_map.Acorn_grouped=="Affluent", ["LCLid",'file']]
# adversity_households = lclid_acorn_map.loc[lclid_acorn_map.Acorn_grouped=="Adversity", ["LCLid",'file']]
# comfortable_households = lclid_acorn_map.loc[lclid_acorn_map.Acorn_grouped=="Comfortable", ["LCLid",'file']]


Let's take a subset of the data because if we take everything, it will hit your RAM. Depending on how much RAM you have, we can choose larger data. But to maintain the variety in the dataset, we will do stratified sampling based on Acorn classifications.

* <= 50 households for 4GB RAM
* 50 - 100 households for 8GB RAM
* 100-150 households for 16GB RAM
* 250 households for 32GB RAM

Let's sample 150 households now, but feel free to reduce of increase as per your hardware constraints

150 households means 50 each from the three Acorn Groups - Affluent, Comfortable, Adversity (we are ignoring the households with unknown ACORN groups)

In [5]:
# selected_households = pd.concat(
#     [
#         affluent_households.sample(50, random_state=76),
#         comfortable_households.sample(50, random_state=76),
#         adversity_households.sample(50, random_state=76),
#     ]
# )
# selected_households['block']=selected_households.file.str.split("_", expand=True).iloc[:,1].astype(int)

In [ ]:
# # extracting the paths to the different blocks and extracting the starting and ending blocks
# path_blocks = [
#     (p, *list(map(int, p.name.split("_")[5].split(".")[0].split("-"))))
#     for p in Path("data/london_smart_meters/preprocessed").glob(
#         "london_smart_meters_merged_block*"
#     )
# ]

In [6]:
# household_df_l = []
# for path, start_b, end_b in tqdm(path_blocks):
#     block_df = pd.read_parquet(path)
#     selected_households['block'].between
#     mask = selected_households['block'].between(start_b, end_b)
#     lclids = selected_households.loc[mask, "LCLid"]
#     household_df_l.append(block_df.loc[block_df.LCLid.isin(lclids)])

In [8]:
# block_df = pd.concat(household_df_l)
# del household_df_l
# block_df.head()

In [ ]:
# from src.utils.data_utils import compact_to_expanded

In [9]:
# #Converting to expanded form
# exp_block_df = compact_to_expanded(block_df, timeseries_col = 'energy_consumption',
# static_cols = ["frequency", "series_length", "stdorToU", "Acorn", "Acorn_grouped", "file"],
# time_varying_cols = ['holidays', 'visibility', 'windBearing', 'temperature', 'dewPoint',
#        'pressure', 'apparentTemperature', 'windSpeed', 'precipType', 'icon',
#        'humidity', 'summary'],
# ts_identifier = "LCLid")

# exp_block_df.head()

## Reduce Memory Footprint

In [10]:
# from src.utils.data_utils import reduce_memory_footprint

In [11]:
# exp_block_df.info(memory_usage="deep", verbose=False)

In [ ]:
# exp_block_df = reduce_memory_footprint(exp_block_df)

In [12]:
# exp_block_df.info(memory_usage="deep", verbose=False)

# Train Test Valildation Split

We are going to keep 2014 data as the validation and test period. We have 2 months(Jan and Feb) of data in 2014. Jan is Validation and Feb is Test

In [18]:
exp_block_df = pd.read_csv(r"data\jepxSpot.csv", index_col = 0, parse_dates = True)[["Kyushu Yen/kWh"]].reset_index().rename(columns = {"datetime": "timestamp", "Kyushu Yen/kWh": "spot_price"})

In [24]:
val_mask = ((exp_block_df.timestamp.dt.year == 2024) & (exp_block_df.timestamp.dt.month >= 4)) | \
           ((exp_block_df.timestamp.dt.year == 2025) & (exp_block_df.timestamp.dt.month <= 3))

test_mask = exp_block_df.timestamp >= pd.Timestamp("2025-04-01")

train = exp_block_df[~(val_mask | test_mask)]
val = exp_block_df[val_mask]
test = exp_block_df[test_mask]

print(f"# of Training samples: {len(train)} | # of Validation samples: {len(val)} | # of Test samples: {len(test)}")
print(f"Max Date in Train: {train.timestamp.max()} | Min Date in Validation: {val.timestamp.min()} | Min Date in Test: {test.timestamp.min()}")

# of Training samples: 245472 | # of Validation samples: 17520 | # of Test samples: 22656
Max Date in Train: 2024-03-31 23:30:00 | Min Date in Validation: 2024-04-01 00:00:00 | Min Date in Test: 2025-04-01 00:00:00


In [ ]:
# train.to_parquet("data/london_smart_meters/preprocessed/selected_blocks_train.parquet")
# val.to_parquet("data/london_smart_meters/preprocessed/selected_blocks_val.parquet")
# test.to_parquet("data/london_smart_meters/preprocessed/selected_blocks_test.parquet")

## Train Test Split after filling in missing values

In [21]:
# from src.imputation.interpolation import SeasonalInterpolation

# block_df.energy_consumption = block_df.energy_consumption.progress_apply(lambda x: SeasonalInterpolation(seasonal_period=48*7).fit_transform(x.reshape(-1,1)).squeeze())

In [20]:
# #Converting to expanded form
# exp_block_df = compact_to_expanded(block_df, timeseries_col = 'energy_consumption',
# static_cols = ["frequency", "series_length", "stdorToU", "Acorn", "Acorn_grouped", "file"],
# time_varying_cols = ['holidays', 'visibility', 'windBearing', 'temperature', 'dewPoint',
#        'pressure', 'apparentTemperature', 'windSpeed', 'precipType', 'icon',
#        'humidity', 'summary'],
# ts_identifier = "LCLid")

# exp_block_df.head()

## Reduce Memory Footprint

In [ ]:
# from src.utils.data_utils import reduce_memory_footprint

In [ ]:
# exp_block_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4711440 entries, 0 to 33263
Columns: 21 entries, timestamp to summary
dtypes: datetime64[ns](1), float64(8), int64(2), object(10)
memory usage: 3.3 GB


In [ ]:
# exp_block_df = reduce_memory_footprint(exp_block_df)

In [ ]:
# exp_block_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4711440 entries, 0 to 33263
Columns: 21 entries, timestamp to summary
dtypes: category(10), datetime64[ns](1), float32(8), int32(2)
memory usage: 301.1 MB


In [22]:
test_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==2)
val_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==1)

train = exp_block_df[~(val_mask|test_mask)]
val = exp_block_df[val_mask]
test = exp_block_df[test_mask]
print(f"# of Training samples: {len(train)} | # of Validation samples: {len(val)} | # of Test samples: {len(test)}")
print(f"Max Date in Train: {train.timestamp.max()} | Min Date in Validation: {val.timestamp.min()} | Min Date in Test: {test.timestamp.min()}")

# of Training samples: 282816 | # of Validation samples: 1488 | # of Test samples: 1344
Max Date in Train: 2026-07-16 23:30:00 | Min Date in Validation: 2014-01-01 00:00:00 | Min Date in Test: 2014-02-01 00:00:00


In [92]:
train.to_parquet("data/london_smart_meters/preprocessed/selected_blocks_train_missing_imputed.parquet")
val.to_parquet("data/london_smart_meters/preprocessed/selected_blocks_val_missing_imputed.parquet")
test.to_parquet("data/london_smart_meters/preprocessed/selected_blocks_test_missing_imputed.parquet")